# テーブルにビュー機能が来た——SnowflakeのVirtual Columnsで売上・利益を計算列として直接定義する — 検証ノートブック

## このノートブックについて

Zenn 記事「[テーブルにビュー機能が来た——SnowflakeのVirtual Columnsで売上・利益を計算列として直接定義する](https://zenn.dev/gtk0326/articles/i68-virtual-columns-general-availability)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## ステップ1: 仮想列を持つテーブルを作成してデータを確認する


In [ ]:
-- 売上・利益を仮想列として定義
CREATE OR REPLACE TABLE sales (
  product_name VARCHAR,
  unit_price   INT,
  quantity     INT,
  cost_price   INT,
  revenue      INT AS (unit_price * quantity),
  profit       INT AS ((unit_price - cost_price) * quantity)
);

-- 利益率を後から追加
ALTER TABLE sales ADD COLUMN profit_margin FLOAT AS (ROUND((unit_price - cost_price) / unit_price * 100, 1));

-- 物理列にのみ値を挿入する
INSERT INTO sales (product_name, unit_price, quantity, cost_price)
VALUES
  ('商品A', 1500, 10,  800),
  ('商品B', 3000,  4, 1200),
  ('商品C',  800, 25,  300);

SELECT product_name, unit_price, quantity, revenue, profit, profit_margin FROM sales;

## ステップ2: DESCRIBE TABLE で仮想列のメタデータを確認する


In [ ]:
DESCRIBE TABLE sales;

## ステップ3: 仮想列を別の仮想列から連鎖参照する

先に定義した仮想列を後の仮想列で参照できます。税込み金額を小計から算出する例です。


In [ ]:
CREATE OR REPLACE TABLE order_summary (
  unit_price     INT,
  quantity       INT,
  subtotal       INT   AS (unit_price * quantity),
  total_with_tax FLOAT AS (subtotal * 1.1)   -- 仮想列 subtotal を参照
);

INSERT INTO order_summary (unit_price, quantity) VALUES (1000, 3);

SELECT unit_price, quantity, subtotal, total_with_tax FROM order_summary;

## ステップ4: 文字列関数・型変換関数を仮想列で使う


In [ ]:
CREATE OR REPLACE TABLE product_report (
  product_name  VARCHAR,
  unit_price    INT,
  quantity      INT,
  revenue       INT     AS (unit_price * quantity),
  revenue_label VARCHAR AS (CONCAT(product_name, ' の売上: ¥', TO_VARCHAR(unit_price * quantity)))
);

INSERT INTO product_report (product_name, unit_price, quantity)
VALUES ('商品A', 1500, 10);

SELECT product_name, revenue, revenue_label FROM product_report;

## ステップ5: 仮想列に対する制約を確認する（失敗ケース）

非決定論的関数（RANDOM など）は使用できません。


In [ ]:
CREATE OR REPLACE TABLE test_nd (
  id       INT,
  rand_val FLOAT AS (RANDOM())
);

-- 仮想列 revenue に値を直接挿入しようとする
INSERT INTO sales (product_name, unit_price, quantity, cost_price, revenue)
VALUES ('商品D', 2000, 5, 1000, 99999);

DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS order_summary;
DROP TABLE IF EXISTS product_report;
DROP TABLE IF EXISTS test_nd;

## クリーンアップ

検証で作成したオブジェクトをすべて削除してください。

> **必ず実行してください。** Dynamic Table などを残すとバックグラウンドでリフレッシュが継続しクレジットが消費されます。

In [ ]:
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS order_summary;
DROP TABLE IF EXISTS product_report;
DROP TABLE IF EXISTS test_nd;